> - coder: yangdong
> - date: 260515
> - tutorial: https://github.com/gao-lab/SLAT/tree/main/docs/tutorials
> - image: SLAT

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

import scSLAT
from scSLAT.model import Cal_Spatial_Net, load_anndatas, run_SLAT, spatial_match
from scSLAT.viz import match_3D_multi, hist, Sankey

In [ ]:
sc.set_figure_params(scanpy=True, dpi=100, dpi_save=150, frameon=True, vector_friendly=True, fontsize=14)

In [ ]:
h5ad1_path=
h5ad2_path=
cluster_key="spatial.cluster"

In [ ]:
adata1 = sc.read_h5ad(h5ad1_path); print(adata1)
adata2 = sc.read_h5ad(h5ad2_path); print(adata2)

In [ ]:

sc.pl.spatial(adata1, color=cluster_key, spot_size=10)
sc.pl.spatial(adata2, color=cluster_key, spot_size=10)

## run SLAT

SLAT need to build neighbor graphs based on cell-cell distance of every dataset respectively.

In [ ]:

Cal_Spatial_Net(adata1, k_cutoff=10, model='KNN')
Cal_Spatial_Net(adata2, k_cutoff=10, model='KNN')

Then, SLAT extract cell gene expression features by a SVD-based matrix factorization algorithm (we named it 'DPCA'), and extract edges of graphs we built in previous step.

> NOTE: SLAT support three built-in embedding algorithms (`DDPCA`, `Harmony`, and `PCA`) currently. But you can use any embedding method manually.

In [ ]:

edges, features = load_anndatas([adata1, adata2], feature='DPCA')

In [ ]:

embd0, embd1, time = run_SLAT(features, edges)

## aligning and visualization

In [ ]:

best, index, distance = spatial_match([embd0, embd1], adatas=[adata1,adata2], reorder=False)

In [ ]:

adata1_df = pd.DataFrame({'index': range(embd0.shape[0]),
                        'x': adata1.obsm['spatial'][:,0],
                        'y': adata1.obsm['spatial'][:,1],
                        'celltype': adata1.obs[cluster_key]})
adata2_df = pd.DataFrame({'index': range(embd1.shape[0]),
                        'x': adata2.obsm['spatial'][:,0],
                        'y': adata2.obsm['spatial'][:,1],
                        'celltype': adata2.obs[cluster_key]})

matching = np.array([range(index.shape[0]), best])
best_match = distance[:,0]

Then we visualizze the cell to cell matching, colored by cell type. By default, Blue line means correct match of cell type, red line is the opposite.

In [ ]:

multi_align = match_3D_multi(adata1_df, adata2_df, matching,meta='celltype',
                            scale_coordinate=True, subsample_size=300)
multi_align.draw_3D(size=[7, 8], line_width=1, point_size=[1.5,1.5], hide_axis=True)

## similarity score

Similarity score means the confident of SLAT alignment. Regions with low similarity scores may 1)have biological difference; 2)cause by technology variance. We can also plot the distribution of the similarity score of aligned cells.

In [ ]:

%matplotlib inline
hist(best_match, cut=0.8)

In [ ]:

adata2.obs['low_quality_index'] = best_match 
adata2.obs['low_quality_index'] = adata2.obs['low_quality_index'].astype(float)

In [ ]:

sc.pl.spatial(adata2, color='low_quality_index', spot_size=20, title='Quality')

## cell type level analysis

We can check the cel type level corresponding via Sankey diagram.

In [ ]:

adata2_df['target_celltype'] = adata1_df.iloc[matching[1,:],:]['celltype'].to_list()
matching_table = adata2_df.groupby(['celltype','target_celltype']).size().unstack(fill_value=0)

In [ ]:

Sankey(matching_table, prefix=['E15.5_E1S1','E15.5_E1S2'])